<a href="https://colab.research.google.com/github/Trangnguyen1402/build-ai-agents-and-chatbots-with-langgraph-2021112/blob/main/Product_Suggestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 01. Setup Gemini Model


In [11]:
!pip install -qU langchain-google-genai langgraph langchain-core \
langchain-community langchain-text-splitters pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompa

## 02. Connect to Gemini API

In [12]:
from google.colab import userdata
import os

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

api_key = userdata.get("GEMINI_API_KEY").strip()
os.environ["GOOGLE_API_KEY"] = api_key

model = ChatGoogleGenerativeAI(
    model="gemini-3.7-flash",
    temperature=1.0
)

embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)

print("Gemini setup successful!")

Gemini setup successful!


## 03. Download Product Data Files

In [13]:
!wget -q -O "/content/Laptop pricing.csv" \
"https://raw.githubusercontent.com/LinkedInLearning/build-ai-agents-and-chatbots-with-langgraph-2021112/main/data/Laptop%20pricing.csv"

!wget -q -O "/content/Laptop product descriptions.pdf" \
"https://raw.githubusercontent.com/LinkedInLearning/build-ai-agents-and-chatbots-with-langgraph-2021112/main/data/Laptop%20product%20descriptions.pdf"

!ls -lh "/content/Laptop pricing.csv" \
"/content/Laptop product descriptions.pdf"

-rw-r--r-- 1 root root  132 Oct 19 06:34 '/content/Laptop pricing.csv'
-rw-r--r-- 1 root root 1.9K Oct 19 06:34 '/content/Laptop product descriptions.pdf'


## 04. Create Product Pricing Tools

In [14]:
import pandas as pd
from langchain_core.tools import tool

product_pricing_df = pd.read_csv("/content/Laptop pricing.csv")

print(product_pricing_df)


@tool
def get_laptop_price(laptop_name: str) -> int:
    """
    Returns the price of a laptop.
    Returns -1 if the laptop is not found.
    """
    matches = product_pricing_df[
        product_pricing_df["Name"].str.contains(
            laptop_name,
            case=False,
            na=False
        )
    ]

    if len(matches) == 0:
        return -1

    return int(matches["Price"].iloc[0])


@tool
def list_available_laptops() -> str:
    """
    Returns all available laptops, prices, and shipping times.
    """
    return product_pricing_df.to_string(index=False)


print("Pricing tools created successfully!")

            Name  Price  ShippingDays
0  AlphaBook Pro   1499             2
1     GammaAir X   1399             7
2  SpectraBook S   2499             7
3   OmegaPro G17   2199            14
4  NanoEdge Flex   1699             2
Pricing tools created successfully!


## 05. Create Product Features Retrieval Tool

In [15]:
from langchain_core.tools import create_retriever_tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(
    "/content/Laptop product descriptions.pdf"
)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=256
)

splits = text_splitter.split_documents(docs)

prod_feature_store = InMemoryVectorStore.from_documents(
    documents=splits,
    embedding=embedding
)

get_product_features = create_retriever_tool(
    prod_feature_store.as_retriever(
        search_kwargs={"k": 2}
    ),
    name="Get_Product_Features",
    description="""
    This tool contains laptop information, including processors,
    memory, storage, design, and advantages.
    Use this tool to find laptop features.
    """
)

print("Product features tool created successfully!")

/tmp/ipykernel_1224/978946379.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Product features tool created successfully!


## 06. Create the Product Recommendation Agent

In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

recommendation_system_prompt = """
You are a helpful Product Recommendation Agent for an online laptop store.

Your job is to recommend laptops based on the customer's:
- Budget
- Intended use
- Performance needs
- Portability preferences

Rules:
1. Use the available tools to check prices and features.
2. Never invent laptop information.
3. Ask follow-up questions when necessary.
4. Recommend one or two laptops.
5. Explain why each recommendation fits the customer.
6. Clearly state the laptop name and price.
7. Be friendly and professional.
"""

recommendation_tools = [
    list_available_laptops,
    get_laptop_price,
    get_product_features
]

recommendation_memory = MemorySaver()

product_recommendation_agent = create_agent(
    model=model,
    tools=recommendation_tools,
    system_prompt=recommendation_system_prompt,
    checkpointer=recommendation_memory
)

print("Product Recommendation Agent created successfully!")

Product Recommendation Agent created successfully!


## 07. Test the Product Recommendation Agent

In [17]:
import uuid

config = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    }
}

questions = [
    "I need a laptop for school.",
    "My budget is around $1,500.",
    "I want something portable."
]


def clean_text(content):
    if isinstance(content, list):
        return "\n".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict) and "text" in block
        )

    return str(content)


for question in questions:
    response = product_recommendation_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        },
        config=config
    )

    content = response["messages"][-1].content
    answer = clean_text(content)

    print("\n" + "-" * 60)
    print(f"USER: {question}")
    print(f"AGENT: {answer}")


------------------------------------------------------------
USER: I need a laptop for school.
AGENT: Hello! I would be happy to help you find the right laptop for your studies. 

For school, portability, reliable performance, and battery-friendly designs are usually top priorities. Here are two great options that fit students well:

---

### 1. **GammaAir X** — **$1,399**
* **Key Specs:** AMD Ryzen 7 processor, 32GB DDR4 RAM, 512GB NVMe SSD.
* **Why it fits:** It features a thin and lightweight design that is easy to carry across campus between classes. With 32GB of RAM and a powerful Ryzen 7 processor, it handles heavy multitasking—like having dozens of research tabs, documents, and lecture streams open at once—without slowing down.

---

### 2. **AlphaBook Pro** — **$1,499**
* **Key Specs:** 12th Gen Intel Core i7 processor, 16GB DDR4 RAM, 1TB SSD.
* **Why it fits:** This sleek ultrabook offers an excellent balance of power and portability. The generous 1TB of fast solid-state stor

## 08. Test Conversation Memory with Two Users

In [18]:
def execute_prompt(user, config, prompt):
    response = product_recommendation_agent.invoke(
        {
            "messages": [
                ("user", prompt)
            ]
        },
        config=config
    )

    content = response["messages"][-1].content
    answer = clean_text(content)

    print(f"\n{user}: {answer}")


# Create separate conversation threads
config_1 = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    }
}

config_2 = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    }
}


# User 1 conversation
execute_prompt(
    "USER 1",
    config_1,
    "I need a laptop for school."
)

execute_prompt(
    "USER 1",
    config_1,
    "My budget is $1,500 and I want something portable."
)


# User 2 conversation
execute_prompt(
    "USER 2",
    config_2,
    "I need a laptop for gaming."
)

execute_prompt(
    "USER 2",
    config_2,
    "My budget is $2,200 and I need strong performance."
)


USER 1: To help you find the perfect laptop for school, could you share a few more details?

1. **What is your budget?**
2. **What kind of schoolwork will you be doing?** (e.g., general coursework, writing papers, and web browsing, or more demanding tasks like engineering software, programming, or graphic design?)
3. **Do you have any portability or design preferences?** (e.g., lightweight for carrying between classes, a 2-in-1 touchscreen for taking notes, or a larger screen?)

Once I know what you're looking for, I'll be happy to recommend the best options for you!

USER 1: Based on your $1,500 budget and need for portability, here are the two best options from our catalog:

---

### 1. **GammaAir X** – **$1,399**
* **Key Specs:** AMD Ryzen 7 processor, 32GB DDR4 RAM, 512GB NVMe SSD
* **Why it fits you:**
  * **Exceptional Portability:** Designed with a thin and light form factor, making it effortless to slip into your backpack and carry across campus all day.
  * **Multitasking Pow